# 08 - Subagents and multi-agent systems

A subagent is a separate agent instance your main agent can spawn for a focused
subtask. It gets its own context, its own system prompt, and its own tool set.

This is the most over-used idea in agent engineering, so we will be precise
about when it pays and when it does not.

**Docs:** [Subagents](https://code.claude.com/docs/en/agent-sdk/subagents)

In [ ]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AgentDefinition,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

SCRATCH = ROOT / "workspace" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

(SCRATCH / "auth.py").write_text(
    "import hashlib\n\n"
    "USERS = {}\n\n"
    "def register(name, password):\n"
    "    USERS[name] = hashlib.md5(password.encode()).hexdigest()\n\n"
    "def login(name, password):\n"
    "    return USERS.get(name) == hashlib.md5(password.encode()).hexdigest()\n\n"
    "def run_query(db, name):\n"
    "    return db.execute('SELECT * FROM users WHERE name = ' + name)\n",
    encoding="utf-8",
)
print("wrote auth.py (md5 hashing, string-concatenated SQL - deliberate)")

## 8.1 Four reasons to use a subagent

**1. Context isolation.** A subagent's context starts fresh. Its intermediate
tool calls and results stay inside it; only its **final message** returns to the
parent as a tool result. A researcher can read twenty files and the parent's
context grows by one paragraph.

**2. Parallelism.** Independent subtasks run concurrently. Three reviewers
finish in the time of the slowest, not the sum of all three.

**3. Specialisation.** Each subagent has a short prompt about one job. A
security reviewer's prompt can be paranoid precisely because it does not also
have to be a helpful generalist.

**4. Tool restriction.** A subagent given `tools=["Read","Grep"]` cannot edit
anything. Not "will not" — cannot. The tool is not in its session.

Reason 4 is the one that matters most when untrusted text (a customer ticket, a
scraped page, a PR description) enters the loop. Prompt injection can only reach
tools the agent actually has.

## 8.2 Defining subagents

Three ways, and one recommendation.

- **Programmatic** — `agents={...}` in your options. Recommended for SDK apps:
  version-controlled with the code, no filesystem dependency.
- **Filesystem** — markdown files in `.claude/agents/`. Shared with the Claude
  Code CLI. Programmatic definitions win on a name collision.
- **Built-in** — Claude can spawn a `general-purpose` subagent with no
  definition at all.

Claude invokes subagents through the **`Agent`** tool, so `Agent` must be in
`allowed_tools` or every spawn hits the permission flow.

```python
AgentDefinition(
    description="...",   # REQUIRED - Claude reads this to decide when to use it
    prompt="...",        # REQUIRED - the subagent's system prompt
    tools=[...],         # omit to inherit everything
    disallowedTools=[...],
    model="sonnet",      # 'fable' | 'opus' | 'sonnet' | 'haiku' | 'inherit' | full ID
    skills=[...],
    mcpServers=[...],    # by name or inline config
    maxTurns=8,
    background=True,
    effort="low",
    permissionMode="dontAsk",
)
```

> **camelCase.** `disallowedTools`, `mcpServers`, `maxTurns`, `permissionMode`,
> and `initialPrompt` keep their camelCase spelling **in Python**, matching the
> wire format rather than PEP 8. `AgentDefinition` is a dataclass, so
> `max_turns=8` fails loudly with `TypeError: unexpected keyword argument`.
> Annoying the first time; far better than a setting that silently does nothing.
> Use the typed `AgentDefinition` rather than a plain dict for exactly that
> reason.

## 8.3 Two specialists, running in parallel

`description` is what Claude matches against when deciding whom to call, so
write it as a routing rule, not as a job title.

In [ ]:
REVIEWERS = {
    "security-reviewer": AgentDefinition(
        description=(
            "Finds security vulnerabilities in code: injection, weak crypto, "
            "auth flaws, secret handling. Use for any security review."
        ),
        prompt=(
            "You are a security reviewer. Report only real, exploitable issues.\n"
            "For each: the vulnerability, the line, the concrete fix.\n"
            "Rank by severity. No style opinions. Be brief."
        ),
        tools=["Read", "Grep", "Glob"],       # read-only: cannot 'fix' anything
        maxTurns=6,
    ),
    "style-reviewer": AgentDefinition(
        description=(
            "Reviews code for naming, structure, readability, and Python "
            "conventions. Use for style and maintainability reviews."
        ),
        prompt=(
            "You are a Python style reviewer. Comment on naming, structure, "
            "type hints, and docstrings. Ignore security entirely - "
            "another reviewer covers it. Be brief."
        ),
        tools=["Read", "Grep", "Glob"],
        maxTurns=6,
    ),
}

spawned = []

async for m in query(
    prompt=("Review auth.py. Use the security-reviewer and the style-reviewer, "
            "then give me a combined summary with security findings first."),
    options=ClaudeAgentOptions(
        agents=REVIEWERS,
        tools=["Read", "Grep", "Glob", "Agent"],
        allowed_tools=["Read", "Grep", "Glob", "Agent"],   # Agent = allowed to spawn
        permission_mode="dontAsk",
        cwd=str(SCRATCH), setting_sources=[], max_turns=25,
        max_budget_usd=0.60, model=MODEL,
    ),
):
    if isinstance(m, AssistantMessage):
        inside = bool(getattr(m, "parent_tool_use_id", None))
        for b in m.content:
            # Renamed Task -> Agent in Claude Code v2.1.63; match both.
            if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"):
                which = b.input.get("subagent_type", "general-purpose")
                spawned.append(which)
                print(f"   >> spawn {which}")
            elif isinstance(b, TextBlock) and b.text.strip() and not inside:
                print(f"   {b.text.strip()[:400]}")
    elif isinstance(m, ResultMessage):
        print(f"\n   ==> {m.subtype} | {m.num_turns} turns | ${m.total_cost_usd or 0:.4f}")

print("\nsubagents spawned:", spawned)

## 8.4 What a subagent inherits — and what it does not

| Receives | Does **not** receive |
|---|---|
| its own `prompt` (system prompt) | the parent's conversation history |
| the `Agent` tool call's prompt string | the parent's tool results |
| project `CLAUDE.md`, if `setting_sources` allows | the parent's system prompt |
| tool definitions (inherited, or the `tools` subset) | preloaded skills, unless listed in `skills` |

**The practical consequence:** the only thing you pass from parent to subagent is
the prompt string in the `Agent` tool call. File paths, error messages, prior
decisions, the customer's name — if it is not in that string, the subagent does
not have it.

Half of all "my subagent gave a useless answer" reports are this. The subagent
was not confused; it was under-briefed.

The parent receives the subagent's **final message** as the tool result, and may
summarise it further. To keep subagent output verbatim, say so in the parent's
prompt.

## 8.5 Detecting subagent activity

Two signals in the message stream:

- a `ToolUseBlock` with `name == "Agent"` (or `"Task"` on older versions) —
  a spawn. `block.input["subagent_type"]` says which.
- any message with a non-empty `parent_tool_use_id` — produced **inside** a
  subagent.

For structured tracking, use `SubagentStart` / `SubagentStop` hooks instead;
their input carries `agent_type` and `agent_id`.

```python
hooks={"SubagentStop": [HookMatcher(hooks=[track])]}
```

## 8.6 Bounding the tree

Once `Agent` is in `allowed_tools`, **Claude decides** how many subagents to
spawn — and subagents can spawn their own. One prompt can become a tree.

| Limit | Set with | Default |
|---|---|---|
| Depth | `env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1"}` | 3 layers |
| Concurrency | `env={"CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS": "5"}` | 20 at once |
| Spend | `max_budget_usd=5.0` | no limit |

`max_budget_usd` is the one that actually bounds the whole tree: subagent spend
counts toward `total_cost_usd`, and at the cap the SDK refuses new spawns, stops
running background subagents, and ends with `error_max_budget_usd`.

> In Python, `env` is **merged** into the inherited environment. In TypeScript it
> **replaces** it, so you must spread `process.env` to keep `PATH`.

Claude Opus 5 delegates more readily than earlier models, which makes these
limits matter more, not less.

In [ ]:
# Same review, tightly bounded. Depth 1 = our subagents cannot spawn their own.
async for m in query(
    prompt="Use the security-reviewer to check auth.py. Two findings maximum.",
    options=ClaudeAgentOptions(
        agents=REVIEWERS,
        tools=["Read", "Grep", "Glob", "Agent"],
        allowed_tools=["Read", "Grep", "Glob", "Agent"],
        permission_mode="dontAsk",
        cwd=str(SCRATCH), setting_sources=[],
        max_turns=15,
        max_budget_usd=0.25,
        env={
            "CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1",
            "CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS": "2",
        },
        model=MODEL,
    ),
):
    if isinstance(m, ResultMessage):
        print(f"{m.subtype} | {m.num_turns} turns | ${m.total_cost_usd or 0:.4f}")
        # usage counts the MAIN loop only. model_usage includes subagents.
        print("usage (main loop only):", m.usage)
        print("model_usage (whole tree):", m.model_usage)

Look at those two numbers side by side. `usage` covers the top-level
loop; `model_usage` covers the whole tree. As soon as you have subagents, `usage`
undercounts — budget from `total_cost_usd` or `model_usage` instead.

## 8.7 Three orchestration patterns

**Fan-out / fan-in.** Independent specialists in parallel, parent merges.
Reviews, multi-source research, batch classification. *Our support-triage team
does this with `kb-researcher` and `account-analyst`.*

```
        ┌──▶ reviewer A ──┐
parent ─┼──▶ reviewer B ──┼──▶ parent merges
        └──▶ reviewer C ──┘
```

**Pipeline.** Each stage consumes the previous one's output. Research → draft →
review. *Our team does this with `reply-drafter` → `qa-reviewer`.*

```
parent ──▶ research ──▶ draft ──▶ review ──▶ parent commits
```

**Generator / critic.** One agent produces, an adversarial one checks, the
parent decides. Effective precisely because the critic's prompt is not also
trying to be helpful.

For work that coordinates dozens to hundreds of agents, turn-by-turn delegation
stops scaling; see
[dynamic workflows](https://code.claude.com/docs/en/workflows).

## 8.8 When *not* to use subagents

Be honest about the cost. A five-agent team costs roughly three to five times a
single agent on the same task, and takes longer.

**Skip subagents when:**

- The task is one lookup and one answer. Delegation is pure overhead.
- The subtasks are sequential and each needs the full prior context — you will
  spend more re-briefing than you save.
- You cannot write a clear `description`. If you cannot say when to use it,
  Claude cannot either.
- Latency is the product.

**Use them when:**

- Subtasks are genuinely independent (→ parallelism)
- One subtask reads a lot and reports a little (→ context isolation)
- A subtask needs materially different instructions (→ specialisation)
- A subtask should be structurally unable to act (→ tool restriction)

The cheapest reliability win is usually not a bigger team. It is one adversarial
reviewer plus a `PreToolUse` hook.

## 8.9 Exercises

1. Add a `test-writer` subagent with `tools=["Read","Write","Bash"]`. Ask for a
   review *and* tests. Does Claude run them in parallel?
2. Give `security-reviewer` `tools=["Read","Edit","Grep"]` and ask it to "fix
   what you find". Then remove `Edit`. Which version would you deploy?
3. Set `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` to `0` and ask for a review that
   obviously wants delegation. What happens?
4. Deliberately under-brief: prompt only `"Use the security-reviewer"`, with no
   filename. Watch what a fresh context with no file path does.
5. **The real one:** notebooks [10](10_build_support_triage.ipynb) and
   [11](11_build_multi_agent_triage.ipynb) build a single agent and a five-agent
   team on the same support-triage problem, then score both. Run TKT-1008
   through each and decide whether the team earned its cost.

In [ ]:
# Scratch space for the exercises.

## Next

[**09 - Cost, observability, and production**](09_cost_and_production.ipynb) —
what to measure, what to cap, and what to check before this runs unattended.

---

### References

- [Subagents in the SDK](https://code.claude.com/docs/en/agent-sdk/subagents)
- [Claude Code subagents](https://code.claude.com/docs/en/sub-agents)
- [Dynamic workflows](https://code.claude.com/docs/en/workflows)
- [A harness for every task](https://claude.com/blog/a-harness-for-every-task-dynamic-workflows-in-claude-code)